# Fetching Census Survey Data

morpc-census connects to the US Census Bureau API, retrieves survey data, and returns it as a normalized long-format DataFrame ready for analysis. Every request is defined by:

- an **Endpoint** — survey dataset and vintage year
- a **scope** — geographic extent (a region, county, state, etc.)
- a **Group** or a list of **variables** — what to retrieve
- an optional **sumlevel** — resolution within the scope (county, tract, place, etc.)

This notebook walks through a typical workflow from discovery to analysis to saving output.

In [ ]:
from morpc_census import (
    Endpoint, Group, CensusAPI, DimensionTable,
    IMPLEMENTED_ENDPOINTS, get_all_avail_endpoints,
    SCOPES,
)
import pandas as pd

## 1. Endpoints — choosing a survey and year

`IMPLEMENTED_ENDPOINTS` lists every survey/table the package supports. `Endpoint` validates the survey name and vintage year at construction time, so you find out immediately if you've specified something unavailable.

In [ ]:
# Surveys the package supports
IMPLEMENTED_ENDPOINTS

In [ ]:
# Construct an endpoint — raises ValueError for unknown surveys or unavailable years
ep = Endpoint('acs/acs5', 2023)
ep

In [ ]:
# All vintage years available for this survey
ep.vintages

## 2. Groups — browsing available variables

> **Network required** — the cells below make live calls to the Census API.

A *group* is a table within a survey — a collection of related variables. `endpoint.groups` returns every group with its description. Fetched once, then cached on the object.

In [ ]:
# Browse available groups — descriptions keyed by group code
{k: v['description'] for k, v in list(ep.groups.items())[:10]}

In [ ]:
# Construct a Group — validates the code against endpoint.groups
group = Group(ep, 'B01001')
print(group.description)
print(group.universe)

In [ ]:
# Variable codes and labels for this group
{k: v.get('label', '') for k, v in list(group.variables.items())[:8]}

## 3. Scopes — choosing a geographic extent

A scope names the geographic coverage of the request — which places to include. `SCOPES` lists all built-in scopes. Pass a scope key to `CensusAPI` as a string.

In [ ]:
# Available scope keys
list(SCOPES.keys())

## 4. Fetching data

`CensusAPI` validates parameters, builds the request, fetches the data, and transforms it into a normalized long-format table in one step. `api.data` holds the raw wide response from the API; `api.long` is the normalized output.

In [ ]:
# Sex by age for all counties in the 15-county MORPC region
b01001 = CensusAPI(ep, 'region15', group=group)
b01001.data

## 5. The long-format result

`api.long` is the normalized output: one row per geography × variable. Type suffixes are split into separate columns (`estimate`, `moe`), variable labels are extracted from the raw label strings, and Census missing-value codes are converted to `NaN`.

In [ ]:
# Long-format output — one row per geography × variable
b01001.long

In [ ]:
# Column types in the long-format output
b01001.long.dtypes

## 6. GEOIDFQs — parsing geography identifiers

The `GEOIDFQ` column holds fully-qualified geography identifiers (e.g. `0500000US39049`). `api.geoidfqs` parses each one into a `GeoIDFQ` object with typed fields for summary level, variant, and component codes.

In [ ]:
# First three geography IDs parsed as GeoIDFQ objects
b01001.geoidfqs[:3]

In [ ]:
# Fields on a GeoIDFQ object
g = b01001.geoidfqs[0]
print('summary level:', g.sumlevel)
print('geoid:        ', g.geoid)
print('parts:        ', g.parts)
print('string form:  ', str(g))

## 7. Drilling down with sumlevel

The `sumlevel` parameter fetches child geographies within the scope. For example, `sumlevel='tract'` with `scope='franklin'` returns all census tracts in Franklin County.

In [ ]:
# Place of birth for all tracts in Franklin County
b05006_tracts = CensusAPI(ep, 'franklin', group=Group(ep, 'B05006'), sumlevel='tract')
b05006_tracts.long

## 8. Fetching specific variables without a group

When you only need a few variables — possibly from different groups — pass `variables` directly instead of a group. The Census API allows at most 50 fields per request; the package batches automatically when you exceed that limit.

In [ ]:
# Total population, male, and female — fetched directly without a group
pop = CensusAPI(ep, 'region15', variables=['B01001_001E', 'B01001_002E', 'B01001_026E'])
pop.long

## 9. Analyzing with DimensionTable

`DimensionTable` pivots the long-format data into the Census Bureau's published table layout: variable labels as a hierarchical row index, geographies as column headers.

In [ ]:
# Wide format — variable label hierarchy as rows, geography × value type as columns
dim = DimensionTable(b01001.long)
dim.wide()

In [ ]:
# Column percentages relative to the Total row
dim.percent()

## 10. Time series

`DimensionTable` accepts any long-format DataFrame with the standard schema. Concatenating `long` from multiple vintages produces a time-series table where `reference_period` becomes a column dimension.

In [ ]:
# Fetch the same group for an earlier vintage
ep2018 = Endpoint('acs/acs5', 2018)
b01001_2018 = CensusAPI(ep2018, 'region15', group=Group(ep2018, 'B01001'))

# Stack the two years and build a dimension table
long_ts = pd.concat([b01001_2018.long, b01001.long])
DimensionTable(long_ts).wide()

## 11. Saving output

`api.save()` writes three files to the output directory:

- `{name}.long.csv` — the long-format data
- `{name}.schema.yaml` — a frictionless Schema describing every column
- `{name}.resource.yaml` — a frictionless Resource descriptor linking the CSV to its schema

The resource is validated immediately after writing, so any schema mismatch surfaces here.

In [ ]:
import os

b01001.save('./temp_data')

print('filename:', b01001.filename)
print('exists:  ', os.path.exists(f'./temp_data/{b01001.filename}'))

In [ ]:
from frictionless import Resource

Resource(f'temp_data/{b01001.name}.resource.yaml')

In [ ]:
from frictionless import Schema

Schema(f'temp_data/{b01001.name}.schema.yaml')